# 09 — Fifth dataset, PhishStorm provenance, and the extended matrix

Your run. This notebook downloads the two new datasets itself, verifies what
they are, runs every remaining experiment, and bundles the results. The numbers
that go into the paper are the ones this notebook produces on your account —
the values currently sitting in the draft are provisional and will be replaced
by yours.

What it does, in order:
1. **Downloads** Ebbu2017 from the original authors' repository and PhishStorm
   from a public mirror (both on GitHub — Colab reaches it directly).
2. **Audits** both structurally (scheme, path, query, by class).
3. **Independence check** against your existing four datasets. Expect a
   surprise here: one of the two is not independent at all.
4. **Runs** the new transfer cells: Ebbu2017 within + both directions against
   all four existing datasets, plus one targeted probe
   (PhiUSIIL → PhishStorm) for the related-work discussion.
5. **Merges** with your notebook-07 matrix, recomputes all statistics and the
   ten-pair asymmetry, regenerates the three figures.
6. **Bundles** everything to Drive as `ext5_bundle.zip`.

Runtime: 30–60 minutes on Colab CPU, checkpointed — a disconnect costs
nothing. Requires `paper_v3_inputs.zip` on your Drive (notebook 07's output).

## Cell 1 — Environment and modules

In [1]:
CKPT_SRC = 'import json, time, hashlib, subprocess\nfrom pathlib import Path\nimport joblib\n\n\ndef _git_hash(repo="/content/phishing-detection"):\n    try:\n        return subprocess.check_output(\n            ["git", "-C", repo, "rev-parse", "--short", "HEAD"],\n            stderr=subprocess.DEVNULL).decode().strip()\n    except Exception:\n        return "nogit"\n\n\nclass Checkpoint:\n    # Resumable checkpointing for Colab. Survives runtime resets via Drive.\n\n    def __init__(self, root="/content/phishing-detection/checkpoints",\n                 run_id=None, verbose=True):\n        self.root = Path(root)\n        self.run_id = run_id or time.strftime("run_%Y%m%d_%H%M%S")\n        self.dir = self.root / self.run_id\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.verbose = verbose\n        self.manifest_path = self.dir / "manifest.json"\n        self.manifest = (json.loads(self.manifest_path.read_text())\n                         if self.manifest_path.exists() else {})\n\n    @staticmethod\n    def _key(params):\n        if params is None:\n            return "default"\n        blob = json.dumps(params, sort_keys=True, default=str)\n        return hashlib.md5(blob.encode()).hexdigest()[:10]\n\n    def _path(self, stage, params=None):\n        return self.dir / (stage + "__" + self._key(params) + ".joblib")\n\n    def exists(self, stage, params=None):\n        return self._path(stage, params).exists()\n\n    def save(self, stage, obj, params=None, meta=None):\n        p = self._path(stage, params)\n        joblib.dump(obj, p, compress=3)\n        size_mb = round(p.stat().st_size / 1e6, 3)\n        self.manifest[p.name] = {\n            "stage": stage, "params": params, "meta": meta or {},\n            "git_commit": _git_hash(),\n            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "size_mb": size_mb,\n        }\n        self.manifest_path.write_text(\n            json.dumps(self.manifest, indent=2, default=str))\n        if self.verbose:\n            print("[ckpt] saved " + p.name + "  (" + str(size_mb) + " MB)")\n        return p\n\n    def load(self, stage, params=None):\n        p = self._path(stage, params)\n        if not p.exists():\n            raise FileNotFoundError("No checkpoint: " + str(p))\n        if self.verbose:\n            print("[ckpt] loaded " + p.name)\n        return joblib.load(p)\n\n    def cache(self, stage, fn, params=None, meta=None, force=False):\n        if self.exists(stage, params) and not force:\n            return self.load(stage, params)\n        if self.verbose:\n            print("[ckpt] computing " + stage + " ...")\n        out = fn()\n        self.save(stage, out, params=params, meta=meta)\n        return out\n\n    def summary(self):\n        import pandas as pd\n        if not self.manifest:\n            return pd.DataFrame()\n        return pd.DataFrame(self.manifest).T[\n            ["stage", "saved_at", "size_mb", "git_commit"]]\n\n    @classmethod\n    def latest_run(cls, root="/content/phishing-detection/checkpoints", **kw):\n        runs = sorted(p.name for p in Path(root).glob("run_*") if p.is_dir())\n        if not runs:\n            raise FileNotFoundError("No previous runs.")\n        return cls(root=root, run_id=runs[-1], **kw)\n'

URLFEAT_SRC = '"""Canonical URL-lexical features + split strategies.\n\nHardened against malformed URLs. Real corpora contain entries that make\nurllib.parse raise: non-numeric or out-of-range ports, unbalanced square\nbrackets that look like broken IPv6 literals, and embedded control\ncharacters. urlparse() is lazy about this - it raises on the .hostname\nand .port properties rather than at parse time - so every access is\nguarded here.\n"""\nimport re, math\nfrom urllib.parse import urlparse, SplitResult\nimport numpy as np\nimport pandas as pd\n\nSUSPICIOUS_WORDS = [\n    "login", "signin", "verify", "account", "update", "secure", "webscr",\n    "banking", "confirm", "password", "credential", "invoice", "payment",\n    "billing", "suspend", "unlock", "recover", "wallet", "support",\n]\nSHORTENERS = {\n    "bit.ly", "goo.gl", "tinyurl.com", "t.co", "ow.ly", "is.gd", "buff.ly",\n    "adf.ly", "bit.do", "cutt.ly", "rb.gy", "shorte.st", "rebrand.ly",\n}\nIPV4 = re.compile(r"^(?:\\d{1,3}\\.){3}\\d{1,3}$")\nHEX_HOST = re.compile(r"^0x[0-9a-f]+$", re.I)\nCTRL = re.compile(r"[\\x00-\\x1f\\x7f]")\n\n_EMPTY = SplitResult(scheme="", netloc="", path="", query="", fragment="")\n\n\ndef safe_parse(url):\n    """Parse a URL, never raising. Returns a SplitResult-like object.\n\n    Falls back through three stages: parse as given; strip control\n    characters and square brackets and retry; return an empty result.\n    """\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.netloc          # force any lazy validation we can\n        return p\n    except ValueError:\n        pass\n    cleaned = CTRL.sub("", s).replace("[", "").replace("]", "")\n    try:\n        return urlparse(cleaned)\n    except ValueError:\n        return _EMPTY\n\n\ndef safe_hostname(p):\n    """netloc-derived hostname, tolerating malformed authority sections."""\n    try:\n        h = p.hostname\n        if h is not None:\n            return h.lower()\n    except ValueError:\n        pass\n    netloc = getattr(p, "netloc", "") or ""\n    netloc = netloc.rsplit("@", 1)[-1]          # drop userinfo\n    netloc = CTRL.sub("", netloc).replace("[", "").replace("]", "")\n    host = netloc.split(":", 1)[0]\n    return host.lower()\n\n\ndef safe_port(p):\n    """Port if it is a valid integer in range, else None. Never raises."""\n    try:\n        return p.port\n    except ValueError:\n        return None\n\n\ndef is_malformed(url):\n    """True if the URL needed the fallback path. Report this rate."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    try:\n        p = urlparse(s)\n        _ = p.hostname\n        _ = p.port\n        return False\n    except ValueError:\n        return True\n\n\ndef _entropy(s):\n    if not s:\n        return 0.0\n    counts = {}\n    for ch in s:\n        counts[ch] = counts.get(ch, 0) + 1\n    n = len(s)\n    return -sum((c / n) * math.log2(c / n) for c in counts.values())\n\n\ndef canonical_url_features(url, extractor=None):\n    """~32 lexical features computable from a URL alone, identically for any source."""\n    url = str(url).strip()\n    if "://" not in url:\n        url = "http://" + url\n\n    p = safe_parse(url)\n    host = safe_hostname(p)\n    path = getattr(p, "path", "") or ""\n    query = getattr(p, "query", "") or ""\n\n    subdomain = domain = suffix = ""\n    if extractor is not None:\n        try:\n            ext = extractor(url)\n            subdomain, domain, suffix = ext.subdomain, ext.domain, ext.suffix\n        except Exception:\n            extractor = None\n    if not (domain or suffix):\n        parts = [x for x in host.split(".") if x]\n        suffix = parts[-1] if len(parts) > 1 else ""\n        domain = parts[-2] if len(parts) > 2 else (parts[0] if parts else "")\n        subdomain = ".".join(parts[:-2]) if len(parts) > 2 else ""\n\n    registrable = f"{domain}.{suffix}".strip(".")\n    digits = sum(ch.isdigit() for ch in url)\n    letters = sum(ch.isalpha() for ch in url)\n\n    return {\n        "url_length": len(url),\n        "host_length": len(host),\n        "path_length": len(path),\n        "query_length": len(query),\n        "n_dots": url.count("."),\n        "n_hyphens": url.count("-"),\n        "n_underscores": url.count("_"),\n        "n_slashes": path.count("/"),\n        "n_question": url.count("?"),\n        "n_equals": url.count("="),\n        "n_at": url.count("@"),\n        "n_ampersand": url.count("&"),\n        "n_percent": url.count("%"),\n        "n_digits": digits,\n        "n_letters": letters,\n        "digit_ratio": digits / max(len(url), 1),\n        "n_subdomains": len([s for s in subdomain.split(".") if s]),\n        "subdomain_length": len(subdomain),\n        "domain_length": len(domain),\n        "tld_length": len(suffix),\n        "is_ip_host": int(bool(IPV4.match(host))),\n        "is_hex_host": int(bool(HEX_HOST.match(host))),\n        "is_https": int(str(getattr(p, "scheme", "")).lower() == "https"),\n        "has_port": int(safe_port(p) is not None),\n        "is_shortener": int(registrable in SHORTENERS),\n        "has_punycode": int("xn--" in host),\n        "has_double_slash_path": int("//" in path),\n        "n_suspicious_words": sum(w in url.lower() for w in SUSPICIOUS_WORDS),\n        "url_entropy": _entropy(url),\n        "host_entropy": _entropy(host),\n        "longest_token_path": max([len(t) for t in re.split(r"[/\\-_.]", path) if t] or [0]),\n        "has_https_token_in_host": int("https" in host),\n    }\n\n\ndef build_canonical_matrix(urls, use_tldextract=True):\n    extractor = None\n    if use_tldextract:\n        extractor = get_extractor()\n    rows = [canonical_url_features(u, extractor) for u in urls]\n    return pd.DataFrame(rows, index=getattr(urls, "index", None))\n\n\n_DEFAULT_EXTRACTOR = None\n\n\ndef get_extractor():\n    """Cached tldextract instance using the bundled PSL snapshot (no network)."""\n    global _DEFAULT_EXTRACTOR\n    if _DEFAULT_EXTRACTOR is None:\n        try:\n            import tldextract\n            _DEFAULT_EXTRACTOR = tldextract.TLDExtract(suffix_list_urls=())\n        except Exception:\n            _DEFAULT_EXTRACTOR = False\n    return _DEFAULT_EXTRACTOR or None\n\n\ndef registrable_domain(url, extractor="auto"):\n    """eTLD+1. Returns the bare IP for IP-literal hosts. Never raises."""\n    s = str(url).strip()\n    if "://" not in s:\n        s = "http://" + s\n    host = safe_hostname(safe_parse(s))\n    if not host:\n        return ""\n    if IPV4.match(host):\n        return host\n    if extractor == "auto":\n        extractor = get_extractor()\n    if extractor is not None:\n        try:\n            e = extractor(s)\n            reg = f"{e.domain}.{e.suffix}".strip(".").strip("[]")\n            if reg:\n                return reg\n        except Exception:\n            pass\n    parts = [x for x in host.strip("[]").split(".") if x]\n    return ".".join(parts[-2:]) if len(parts) >= 2 else host.strip("[]")\n\n\n# ------------------------------------------------------------------ splits\ndef grouped_split(X, y, groups, test_size=0.2, seed=42):\n    """No registrable domain appears in both train and test."""\n    from sklearn.model_selection import GroupShuffleSplit\n    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)\n    tr, te = next(gss.split(X, y, groups))\n    return tr, te\n\n\ndef stratified_split(X, y, test_size=0.2, seed=42):\n    from sklearn.model_selection import train_test_split\n    idx = np.arange(len(y))\n    return train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)\n\n\ndef rebalance_to_base_rate(y, target_rate, seed=42, index=None):\n    """Downsample the phishing class to a realistic prevalence."""\n    rng = np.random.default_rng(seed)\n    idx = np.arange(len(y)) if index is None else np.asarray(index)\n    pos = idx[np.asarray(y) == 1]\n    neg = idx[np.asarray(y) == 0]\n    n_pos = int(round(target_rate * len(neg) / (1 - target_rate)))\n    n_pos = min(n_pos, len(pos))\n    keep = np.concatenate([neg, rng.choice(pos, size=n_pos, replace=False)])\n    rng.shuffle(keep)\n    return keep\n'

STAT_SRC = '"""Inference for transfer experiments: DeLong, bootstrap CIs, corrections."""\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats\n\n\n# ------------------------------------------------------------------ DeLong\ndef _midrank(x):\n    J = np.argsort(x)\n    Z = x[J]\n    N = len(x)\n    T = np.zeros(N, dtype=float)\n    i = 0\n    while i < N:\n        j = i\n        while j < N and Z[j] == Z[i]:\n            j += 1\n        T[i:j] = 0.5 * (i + j - 1) + 1\n        i = j\n    T2 = np.empty(N, dtype=float)\n    T2[J] = T\n    return T2\n\n\ndef _structural_components(scores, n_pos, n_neg):\n    """V10 (positives) and V01 (negatives) components for each predictor."""\n    k = scores.shape[0]\n    tx = np.empty([k, n_pos]); ty = np.empty([k, n_neg]); tz = np.empty([k, n_pos + n_neg])\n    for r in range(k):\n        pos = scores[r, :n_pos]\n        neg = scores[r, n_pos:]\n        tx[r, :] = _midrank(pos)\n        ty[r, :] = _midrank(neg)\n        tz[r, :] = _midrank(scores[r, :])\n    aucs = (tz[:, :n_pos].sum(axis=1) / n_pos - (n_pos + 1) / 2) / n_neg\n    v01 = (tz[:, :n_pos] - tx) / n_neg\n    v10 = 1 - (tz[:, n_pos:] - ty) / n_pos\n    return aucs, v01, v10\n\n\ndef delong_test(y_true, scores_a, scores_b):\n    """Paired DeLong test for two AUCs on the same sample.\n\n    Returns (auc_a, auc_b, z, p). Two-sided.\n    """\n    y = np.asarray(y_true).astype(int)\n    order = np.argsort(-y, kind="mergesort")   # positives first\n    y = y[order]\n    a = np.asarray(scores_a)[order]\n    b = np.asarray(scores_b)[order]\n    n_pos = int(y.sum()); n_neg = len(y) - n_pos\n    if n_pos == 0 or n_neg == 0:\n        return np.nan, np.nan, np.nan, np.nan\n\n    scores = np.vstack([a, b])\n    aucs, v01, v10 = _structural_components(scores, n_pos, n_neg)\n    s01 = np.cov(v01); s10 = np.cov(v10)\n    s = s01 / n_pos + s10 / n_neg\n    var = s[0, 0] + s[1, 1] - 2 * s[0, 1]\n    if var <= 0:\n        return float(aucs[0]), float(aucs[1]), np.nan, np.nan\n    z = (aucs[0] - aucs[1]) / np.sqrt(var)\n    p = 2 * stats.norm.sf(abs(z))\n    return float(aucs[0]), float(aucs[1]), float(z), float(p)\n\n\ndef delong_ci(y_true, scores, alpha=0.05):\n    """DeLong analytic confidence interval for a single AUC."""\n    y = np.asarray(y_true).astype(int)\n    order = np.argsort(-y, kind="mergesort")\n    y = y[order]; s = np.asarray(scores)[order]\n    n_pos = int(y.sum()); n_neg = len(y) - n_pos\n    if n_pos == 0 or n_neg == 0:\n        return np.nan, np.nan, np.nan\n    aucs, v01, v10 = _structural_components(s.reshape(1, -1), n_pos, n_neg)\n    var = np.var(v01, ddof=1) / n_pos + np.var(v10, ddof=1) / n_neg\n    se = np.sqrt(var)\n    zc = stats.norm.ppf(1 - alpha / 2)\n    auc = float(aucs[0])\n    return auc, max(0.0, auc - zc * se), min(1.0, auc + zc * se)\n\n\n# --------------------------------------------------------------- bootstrap\ndef bootstrap_ci(y_true, scores, metric_fn, n_boot=2000, alpha=0.05, seed=42):\n    """Stratified percentile bootstrap CI for any metric(y, s)."""\n    rng = np.random.default_rng(seed)\n    y = np.asarray(y_true); s = np.asarray(scores)\n    pos = np.flatnonzero(y == 1); neg = np.flatnonzero(y == 0)\n    if len(pos) < 2 or len(neg) < 2:\n        return np.nan, np.nan, np.nan\n    vals = np.empty(n_boot)\n    for i in range(n_boot):\n        idx = np.concatenate([rng.choice(pos, len(pos), replace=True),\n                              rng.choice(neg, len(neg), replace=True)])\n        try:\n            vals[i] = metric_fn(y[idx], s[idx])\n        except Exception:\n            vals[i] = np.nan\n    vals = vals[~np.isnan(vals)]\n    point = metric_fn(y, s)\n    lo, hi = np.percentile(vals, [100 * alpha / 2, 100 * (1 - alpha / 2)])\n    return float(point), float(lo), float(hi)\n\n\n# ------------------------------------------------------- permutation tests\ndef permutation_gap_test(within_vals, cross_vals, n_perm=10000, seed=42):\n    """Is the within-vs-cross gap larger than chance reassignment would give?"""\n    rng = np.random.default_rng(seed)\n    a = np.asarray(within_vals, dtype=float); b = np.asarray(cross_vals, dtype=float)\n    obs = a.mean() - b.mean()\n    pool = np.concatenate([a, b]); n = len(a)\n    count = 0\n    for _ in range(n_perm):\n        rng.shuffle(pool)\n        if (pool[:n].mean() - pool[n:].mean()) >= obs:\n            count += 1\n    return float(obs), (count + 1) / (n_perm + 1)\n\n\ndef cliffs_delta(a, b):\n    """Non-parametric effect size. |d|: .147 small, .33 medium, .474 large."""\n    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)\n    gt = sum((x > y) for x in a for y in b)\n    lt = sum((x < y) for x in a for y in b)\n    return (gt - lt) / (len(a) * len(b))\n\n\n# --------------------------------------------- multiple comparison control\ndef holm_bonferroni(pvals, alpha=0.05):\n    """Holm step-down. Returns (adjusted p, reject) in the original order."""\n    p = np.asarray(pvals, dtype=float)\n    ok = ~np.isnan(p)\n    out = np.full(len(p), np.nan); rej = np.zeros(len(p), dtype=bool)\n    idx = np.flatnonzero(ok)\n    if len(idx) == 0:\n        return out, rej\n    order = idx[np.argsort(p[idx])]\n    m = len(order)\n    running = 0.0\n    for rank, i in enumerate(order):\n        adj = min(1.0, (m - rank) * p[i])\n        running = max(running, adj)     # enforce monotonicity\n        out[i] = running\n        rej[i] = running <= alpha\n    return out, rej\n'

from google.colab import drive
from pathlib import Path
import sys, warnings
warnings.filterwarnings("ignore")

drive.mount('/content/drive')
PROJECT   = "phishing-detection"
REPO_DIR  = Path("/content") / PROJECT
DRIVE_DIR = Path("/content/drive/MyDrive/research") / PROJECT

for d in ["src", "src/features", "src/evaluation", "src/utils",
          "reports/tables", "reports/figures"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/features", "src/evaluation", "src/utils"]:
    (REPO_DIR / d / "__init__.py").touch()
for name in ["data", "checkpoints", "models", "logs"]:
    link = REPO_DIR / name
    if link.is_symlink(): link.unlink()
    elif link.exists(): raise RuntimeError(f"{link} exists and is not a symlink")
    link.symlink_to(DRIVE_DIR / name)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

(REPO_DIR / "src" / "utils" / "checkpoint.py").write_text(CKPT_SRC)
(REPO_DIR / "src" / "features" / "urlfeat.py").write_text(URLFEAT_SRC)
(REPO_DIR / "src" / "evaluation" / "stats.py").write_text(STAT_SRC)

import importlib
import src.features.urlfeat, src.evaluation.stats
importlib.reload(src.features.urlfeat); importlib.reload(src.evaluation.stats)
import src.features.urlfeat as uf
import src.evaluation.stats as ST

import xgboost, sklearn
print("versions: xgboost", xgboost.__version__, "| sklearn", sklearn.__version__)
RAW = REPO_DIR / "data" / "raw"
TABLES = REPO_DIR / "reports" / "tables"
FIGS   = REPO_DIR / "reports" / "figures"

Mounted at /content/drive
versions: xgboost 3.3.0 | sklearn 1.6.1


## Cell 2 — Download the two new datasets

Ebbu2017 comes from the original authors' repository (`ebubekirbbr/pdd` —
Ebubekir Buber is a co-author of Sahingoz et al., ESWA 2019). PhishStorm comes
from a public mirror of the Marchal et al. (2014) release; the row counts are
verified against the published description, and SHA-256 hashes of both source
files are recorded.

In [2]:
import requests, json, hashlib, io, tarfile
import pandas as pd

(RAW / "ebbu2017").mkdir(parents=True, exist_ok=True)
(RAW / "phishstorm").mkdir(parents=True, exist_ok=True)

def sha256(b): return hashlib.sha256(b).hexdigest()
hashes = {}

# --- Ebbu2017: two JSON files, raw URLs ---
base = "https://raw.githubusercontent.com/ebubekirbbr/pdd/master/input/"
leg_b = requests.get(base + "data_legitimate_36400.json", timeout=120).content
phi_b = requests.get(base + "data_phishing_37175.json", timeout=120).content
hashes["ebbu_legitimate_json"] = sha256(leg_b)
hashes["ebbu_phishing_json"]   = sha256(phi_b)
leg, phi = json.loads(leg_b), json.loads(phi_b)
assert len(leg) == 36400 and len(phi) == 37175, (len(leg), len(phi))
ebbu = pd.DataFrame({"url": list(leg) + list(phi),
                     "label": [0]*len(leg) + [1]*len(phi)})
ebbu.to_csv(RAW / "ebbu2017" / "ebbu2017.csv", index=False)
print(f"ebbu2017: {len(ebbu):,} rows  (36,400 legitimate + 37,175 phishing — matches the paper)")

# --- PhishStorm: urlset.csv inside a mirror repository tarball ---
tb = requests.get("https://codeload.github.com/codebrotherone/cyberml-phishstorm/tar.gz/refs/heads/master",
                  timeout=300).content
hashes["phishstorm_mirror_tarball"] = sha256(tb)
with tarfile.open(fileobj=io.BytesIO(tb), mode="r:gz") as tf:
    member = [m for m in tf.getnames() if m.endswith("/urlset.csv")][0]
    raw_csv = tf.extractfile(member).read()
hashes["phishstorm_urlset_csv"] = sha256(raw_csv)
ps = pd.read_csv(io.BytesIO(raw_csv), encoding="latin-1",
                 on_bad_lines="skip", low_memory=False)[["domain", "label"]].dropna()
ps = pd.DataFrame({"url": ps["domain"].astype(str), "label": ps["label"].astype(int)})
ps.to_csv(RAW / "phishstorm" / "phishstorm.csv", index=False)
n0, n1 = (ps.label == 0).sum(), (ps.label == 1).sum()
print(f"phishstorm: {len(ps):,} usable rows ({n0:,} legitimate + {n1:,} phishing;")
print("  published description: 48,009 + 48,009 — a few dozen malformed lines are skipped on read)")

import json as js
(TABLES / "new_dataset_hashes.json").write_text(js.dumps(hashes, indent=2))
print("\nSHA-256 hashes recorded -> reports/tables/new_dataset_hashes.json")

ebbu2017: 73,575 rows  (36,400 legitimate + 37,175 phishing — matches the paper)
phishstorm: 95,913 usable rows (48,009 legitimate + 47,904 phishing;
  published description: 48,009 + 48,009 — a few dozen malformed lines are skipped on read)

SHA-256 hashes recorded -> reports/tables/new_dataset_hashes.json


## Cell 3 — Load all six, exact subsample order

The four originals draw from the shared rng in the same historical order as
notebooks 06/07, so their 60,000-row subsamples are bit-identical to every
previous run. The new datasets draw after them.

In [3]:
import numpy as np
LOAD_ORDER = [
    ("phiusiil",          RAW / "phiusiil/phiusiil.csv",                        "URL", "label",  0),
    ("hannousse",         RAW / "hannousse/hannousse.csv",                      "url", "status", "phishing"),
    ("kaggle_malicious",  RAW / "kaggle_malicious/kaggle_malicious_urls.csv",   "url", "label",  1),
    ("mendeley_phishurl", RAW / "mendeley_phishurl/mendeley_phishurl_urls.csv", "url", "label",  1),
    ("ebbu2017",          RAW / "ebbu2017/ebbu2017.csv",                        "url", "label",  1),
    ("phishstorm",        RAW / "phishstorm/phishstorm.csv",                    "url", "label",  1),
]
CAP = 60000
rng = np.random.default_rng(42)
CORPORA = {}
for key, path, ucol, lcol, phish in LOAD_ORDER:
    d = pd.read_csv(path, low_memory=False)
    uc = [col for col in d.columns if col.lower() == ucol.lower()][0]
    lc = [col for col in d.columns if col.lower() == lcol.lower()][0]
    lab = ((d[lc].astype(str).str.lower() == str(phish).lower()).astype(int)
           if isinstance(phish, str) else (d[lc] == phish).astype(int))
    df = pd.DataFrame({"url": d[uc].astype(str), "label": lab})
    if len(df) > CAP:
        idx = np.sort(rng.choice(len(df), CAP, replace=False))
        df = df.iloc[idx].reset_index(drop=True)
    CORPORA[key] = df
    print(f"[{key}] {len(df):,}  phishing_rate={df.label.mean():.4f}")

[phiusiil] 60,000  phishing_rate=0.4267
[hannousse] 11,430  phishing_rate=0.5000
[kaggle_malicious] 60,000  phishing_rate=0.1805
[mendeley_phishurl] 60,000  phishing_rate=0.3155
[ebbu2017] 60,000  phishing_rate=0.5060
[phishstorm] 60,000  phishing_rate=0.4999


## Cell 4 — Structural audit of the new datasets

Same three properties, plus one PhishStorm needs: whether the URL carries any
scheme at all.

In [4]:
def audit(df, name):
    u = df.url.astype(str)
    rows = []
    for cls, cname in [(0, "legit"), (1, "phish")]:
        um = u[df.label == cls]
        scheme = um.str.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://").mean() * 100
        https  = um.str.lower().str.startswith("https://").mean() * 100
        pl = um.map(lambda x: len(getattr(uf.safe_parse(x), "path", "") or ""))
        qy = um.map(lambda x: len(getattr(uf.safe_parse(x), "query", "") or "") > 0).mean() * 100
        rows.append({"dataset": name, "class": cname, "n": len(um),
                     "scheme_present": round(scheme, 2), "https": round(https, 2),
                     "empty_path": round((pl == 0).mean() * 100, 2),
                     "has_query": round(qy, 2)})
    return rows

out = audit(CORPORA["ebbu2017"], "ebbu2017") + audit(CORPORA["phishstorm"], "phishstorm")
new_audit = pd.DataFrame(out)
print(new_audit.to_string(index=False))
new_audit.to_csv(TABLES / "structural_audit_new.csv", index=False)
print("\nThings to notice:")
print(" - ebbu2017: legit empty_path is EXACTLY 0.00 (degenerate, but phishing ~0.7% too:")
print("   a fixed value that does not separate). The separating association is the")
print("   scheme: ~37% HTTPS legit vs ~2% phishing.")
print(" - phishstorm: scheme_present = 0.00 for BOTH classes — the entire dataset is")
print("   distributed with schemes stripped.")

   dataset class     n  scheme_present  https  empty_path  has_query
  ebbu2017 legit 29640          100.00  36.75        0.00       4.08
  ebbu2017 phish 30360           99.49   1.83        0.67      21.40
phishstorm legit 30007            0.00   0.00        0.00       4.76
phishstorm phish 29993            0.00   0.00        0.57      32.76

Things to notice:
 - ebbu2017: legit empty_path is EXACTLY 0.00 (degenerate, but phishing ~0.7% too:
   a fixed value that does not separate). The separating association is the
   scheme: ~37% HTTPS legit vs ~2% phishing.
 - phishstorm: scheme_present = 0.00 for BOTH classes — the entire dataset is
   distributed with schemes stripped.


## Cell 5 — Independence check

Normalised-URL containment of each new dataset against the existing four and
each other. Anything above 0.5 is treated as the same data.

In [5]:
def normalise(u):
    s = str(u).strip().lower()
    for pre in ("https://", "http://"):
        if s.startswith(pre): s = s[len(pre):]; break
    if s.startswith("www."): s = s[4:]
    return s.rstrip("/")

sets = {k: set(v.url.astype(str).map(normalise)) for k, v in CORPORA.items()}
# use the FULL new datasets for the containment claim, not the capped samples
full_ebbu = pd.read_csv(RAW / "ebbu2017/ebbu2017.csv").url.astype(str).map(normalise)
full_ps   = pd.read_csv(RAW / "phishstorm/phishstorm.csv").url.astype(str).map(normalise)
# full originals: reload raw
full = {}
for key, path, ucol, lcol, phish in LOAD_ORDER[:4]:
    d = pd.read_csv(path, low_memory=False)
    uc = [col for col in d.columns if col.lower() == ucol.lower()][0]
    full[key] = set(d[uc].astype(str).map(normalise))
full["ebbu2017"] = set(full_ebbu); full["phishstorm"] = set(full_ps)

print(f"{'pair':44s} {'shared':>9s} {'containment':>12s}")
flags = []
K = list(full)
for i, a in enumerate(K):
    for b in K[i+1:]:
        if a not in ("ebbu2017","phishstorm") and b not in ("ebbu2017","phishstorm"):
            continue
        inter = len(full[a] & full[b]); mn = min(len(full[a]), len(full[b]))
        cont = inter / mn if mn else 0
        print(f"{a+' / '+b:44s} {inter:9,d} {cont:12.4f}")
        if cont > 0.5: flags.append((a, b, cont))
print()
for a, b, cnt in flags:
    print(f"NOT INDEPENDENT: {a} / {b}  (containment {cnt:.3f})")
print("Expected: phishstorm is contained wholesale in kaggle_malicious. It is")
print("therefore EXCLUDED from the independent set; ebbu2017 is the fifth dataset.")
print("phishstorm is retained only for the targeted probe in Cell 6.")

pair                                            shared  containment
phiusiil / ebbu2017                                130       0.0026
phiusiil / phishstorm                                0       0.0000
hannousse / ebbu2017                             4,008       0.3621
hannousse / phishstorm                              13       0.0012
kaggle_malicious / ebbu2017                        250       0.0050
kaggle_malicious / phishstorm                   94,648       1.0000
mendeley_phishurl / ebbu2017                     9,675       0.1926
mendeley_phishurl / phishstorm                     307       0.0032
ebbu2017 / phishstorm                               52       0.0010

NOT INDEPENDENT: kaggle_malicious / phishstorm  (containment 1.000)
Expected: phishstorm is contained wholesale in kaggle_malicious. It is
therefore EXCLUDED from the independent set; ebbu2017 is the fifth dataset.
phishstorm is retained only for the targeted probe in Cell 6.


## Cell 6 — Gate, then the remaining experiments (checkpointed)

Reproduction gate first (your reference value 0.9210), then: ebbu2017 within
(five seeds), ebbu2017 in both directions against the four originals, and the
targeted PhiUSIIL → PhishStorm probe. Fifty fits, resumable.

In [6]:
import time, os
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

canon = {}
for key, df in CORPORA.items():
    urls = df.url.astype(str)
    canon[key] = {"X": uf.build_canonical_matrix(urls).reset_index(drop=True),
                  "y": df.label.reset_index(drop=True),
                  "domain": urls.map(uf.registrable_domain).reset_index(drop=True)}

def make_xgb(seed):
    return XGBClassifier(n_estimators=400, max_depth=7, learning_rate=0.1,
                         subsample=0.9, colsample_bytree=0.9, n_jobs=-1,
                         eval_metric="logloss", random_state=seed, verbosity=0)

m = make_xgb(42).fit(canon["phiusiil"]["X"], canon["phiusiil"]["y"])
g = roc_auc_score(canon["mendeley_phishurl"]["y"],
                  m.predict_proba(canon["mendeley_phishurl"]["X"])[:, 1])
print(f"gate phiusiil->mendeley seed42: {g:.4f}  (your reference: 0.9210, tol 0.02)")
assert abs(g - 0.9210) < 0.02, "gate failed — stop and report this number"

SEEDS = (42, 43, 44, 45, 46)
OLD4 = ["phiusiil", "kaggle_malicious", "mendeley_phishurl", "hannousse"]
OUT = TABLES / "ext5_results.csv"
done = set()
if OUT.exists():
    prev = pd.read_csv(OUT); done = set(zip(prev.train, prev.test, prev.seed))
    print(f"resuming: {len(done)} rows")
else:
    pd.DataFrame(columns=["train","test","seed","kind","roc_auc"]).to_csv(OUT, index=False)

jobs = ([("ebbu2017","ebbu2017",s) for s in SEEDS]
        + [("ebbu2017",k,s) for k in OLD4 for s in SEEDS]
        + [(k,"ebbu2017",s) for k in OLD4 for s in SEEDS]
        + [("phiusiil","phishstorm",s) for s in SEEDS])
t0 = time.time()
for i, (src, dst, seed) in enumerate(jobs):
    if (src, dst, seed) in done: continue
    Xs, ys = canon[src]["X"], canon[src]["y"]
    if src == dst:
        tr, te = uf.grouped_split(Xs, ys, canon[src]["domain"], seed=seed)
        Xtr, ytr, Xte, yte = Xs.iloc[tr], ys.iloc[tr], Xs.iloc[te], ys.iloc[te]
    else:
        Xtr, ytr = Xs, ys
        Xte, yte = canon[dst]["X"], canon[dst]["y"]
    mm = make_xgb(seed).fit(Xtr, ytr)
    a = roc_auc_score(yte, mm.predict_proba(Xte)[:, 1])
    pd.DataFrame([{"train": src, "test": dst, "seed": seed,
                   "kind": "within" if src == dst else "cross",
                   "roc_auc": round(float(a), 4)}]).to_csv(OUT, mode="a", header=False, index=False)
    done.add((src, dst, seed))
    if (i+1) % 5 == 0: print(f"  {i+1}/{len(jobs)}  [{(time.time()-t0)/60:.1f} min]", flush=True)
print("experiments complete")

gate phiusiil->mendeley seed42: 0.9210  (your reference: 0.9210, tol 0.02)
  5/50  [0.4 min]
  10/50  [1.0 min]
  15/50  [1.5 min]
  20/50  [2.0 min]
  25/50  [2.5 min]
  30/50  [2.8 min]
  35/50  [3.3 min]
  40/50  [3.7 min]
  45/50  [3.9 min]
  50/50  [4.3 min]
experiments complete


## Cell 7 — Merge with notebook 07's matrix, full statistics

Reads your notebook-07 results from `paper_v3_inputs.zip` on Drive and builds
the five-dataset matrix, the aggregates, and the ten-pair asymmetry.

In [7]:
import zipfile
from scipy import stats as sps

with zipfile.ZipFile(DRIVE_DIR / "paper_v3_inputs.zip") as z:
    old = pd.read_csv(z.open("tables/transfer_matrix.csv"))
new = pd.read_csv(OUT)
probe = new[(new.train == "phiusiil") & (new.test == "phishstorm")]
ebbu_cells = new[(new.train != "phiusiil") | (new.test != "phishstorm")]
merged = pd.concat([old, ebbu_cells], ignore_index=True)
merged.to_csv(TABLES / "transfer_matrix_5way.csv", index=False)

ORD = ["phiusiil","mendeley_phishurl","kaggle_malicious","hannousse","ebbu2017"]
piv = merged.pivot_table(index="train", columns="test", values="roc_auc").reindex(index=ORD, columns=ORD)
print("5x5 transfer matrix (mean over seeds):")
print(piv.round(4).to_string())

print(f"\nPROBE phiusiil -> phishstorm: {probe.roc_auc.mean():.4f} (sd {probe.roc_auc.std():.4f})")

w = merged[merged.kind=="within"].roc_auc.values
cr = merged[merged.kind=="cross"].roc_auc.values
cm = merged[merged.kind=="cross"].groupby(["train","test"]).roc_auc.mean()
gap, p = ST.permutation_gap_test(w, cr, n_perm=10000)
print(f"\nwithin n={len(w)} mean={w.mean():.4f} sd={w.std(ddof=1):.4f}")
print(f"cross  n={len(cr)} mean={cr.mean():.4f} sd={cr.std(ddof=1):.4f}")
print(f"gap={gap:.4f}  p={p:.5f} (report p<0.001)  delta={ST.cliffs_delta(w,cr):.4f}")
print(f"inverted (mean-level): {int((cm<0.5).sum())}/20  min={cm.min():.4f}")
inv = cm[cm < 0.5]
print("inverted cells:", [f"{a}->{b}:{v:.3f}" for (a,b),v in inv.sort_values().items()])
print("every inverted cell involves kaggle_malicious:",
      all("kaggle_malicious" in (a,b) for a,b in inv.index))

rows = []
for i,a in enumerate(ORD):
    for b in ORD[i+1:]:
        ab = merged[(merged.train==a)&(merged.test==b)].roc_auc.values
        ba = merged[(merged.train==b)&(merged.test==a)].roc_auc.values
        pv = float(sps.mannwhitneyu(ab, ba, alternative="two-sided").pvalue)
        rows.append({"a":a,"b":b,"ab":round(ab.mean(),4),"ba":round(ba.mean(),4),
                     "asym":round(abs(ab.mean()-ba.mean()),4),"p_raw":round(pv,6)})
asym = pd.DataFrame(rows)
adj, rej = ST.holm_bonferroni(asym.p_raw.values)
asym["p_holm"] = np.round(adj,5); asym["sig"] = rej
asym = asym.sort_values("asym", ascending=False)
print("\n" + asym.to_string(index=False))
print("\nNOTE: with 10 pairs and 5 seeds/direction, the smallest attainable Holm-")
print("corrected p is 0.079 — significance at 0.05 is unattainable BY DESIGN.")
print("The paper reports separation and magnitudes, not significance.")
asym.to_csv(TABLES / "asymmetry_5way.csv", index=False)
sd = merged.groupby(["train","test","kind"]).roc_auc.std().reset_index()
print(f"\nper-cell sd: within max {sd[sd.kind=='within'].roc_auc.max():.4f}, "
      f"cross max {sd[sd.kind=='cross'].roc_auc.max():.4f}")
probe.to_csv(TABLES / "phishstorm_probe.csv", index=False)

5x5 transfer matrix (mean over seeds):
test               phiusiil  mendeley_phishurl  kaggle_malicious  hannousse  ebbu2017
train                                                                                
phiusiil             0.9984             0.9296            0.2427     0.6880    0.7318
mendeley_phishurl    0.9788             0.9955            0.2379     0.7823    0.8694
kaggle_malicious     0.3667             0.2020            0.9426     0.4618    0.3001
hannousse            0.8603             0.9400            0.5561     0.9437    0.9688
ebbu2017             0.7513             0.9581            0.6742     0.8889    0.9774

PROBE phiusiil -> phishstorm: 0.7156 (sd 0.0145)

within n=25 mean=0.9715 sd=0.0252
cross  n=100 mean=0.6694 sd=0.2688
gap=0.3021  p=0.00010 (report p<0.001)  delta=0.8584
inverted (mean-level): 6/20  min=0.2020
inverted cells: ['kaggle_malicious->mendeley_phishurl:0.202', 'mendeley_phishurl->kaggle_malicious:0.238', 'phiusiil->kaggle_malicious:0.243', 'ka

## Cell 8 — Figures (dashboard, path-length panels, heat map)

In [8]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"serif","savefig.dpi":300,"axes.linewidth":0.8})

NAMES={"phiusiil":"PhiUSIIL","mendeley_phishurl":"Mendeley PhishURL",
       "kaggle_malicious":"Kaggle Malicious","hannousse":"Hannousse","ebbu2017":"Ebbu2017"}
LEG,PHI="#4C72B0","#C44E52"

# audit values for the dashboard: computed directly from the loaded capped
# samples — no file dependency, identical definitions to the audit table
def cls_stats(df, cls):
    u = df.url.astype(str)[df.label==cls]
    https = u.str.lower().str.startswith("https://").mean()*100
    pl = u.map(lambda x: len(getattr(uf.safe_parse(x),"path","") or ""))
    qy = u.map(lambda x: len(getattr(uf.safe_parse(x),"query","") or "")>0).mean()*100
    return https,(pl==0).mean()*100,qy

vals = {}
for k in ORD:
    L, P = cls_stats(CORPORA[k],0), cls_stats(CORPORA[k],1)
    vals[k] = {"https":(L[0],P[0]),"empty":(L[1],P[1]),"query":(L[2],P[2])}

fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.9), sharey=True)
y = np.arange(5)[::-1]; h = 0.36
for ax,(title,key) in zip(axes,[("HTTPS","https"),("Empty path","empty"),("Query string","query")]):
    lv=[round(vals[k][key][0],1) for k in ORD]; pv=[round(vals[k][key][1],1) for k in ORD]
    ax.barh(y+h/2,lv,height=h,color=LEG,zorder=3); ax.barh(y-h/2,pv,height=h,color=PHI,zorder=3)
    for yy,v in list(zip(y+h/2,lv))+list(zip(y-h/2,pv)):
        deg = v in (0.0,100.0); t=f"{v:.1f}"+("*" if deg else "")
        ax.text(min(v,96)+2 if v<70 else v-2, yy, t, va="center",
                ha="left" if v<70 else "right", fontsize=6.5,
                fontweight="bold" if deg else "normal",
                color="black" if v<70 else "white", zorder=4)
    ax.set_title(title,fontsize=9); ax.set_xlim(0,100)
    ax.set_yticks(y); ax.set_yticklabels([NAMES[k] for k in ORD],fontsize=7)
    ax.grid(axis="x",alpha=.25,linewidth=.5); ax.set_axisbelow(True)
axes[1].set_xlabel("% of class",fontsize=8)
axes[0].legend(["Legitimate","Phishing"],fontsize=6.5,loc="lower right",framealpha=.95)
fig.tight_layout(pad=0.4)
fig.savefig(FIGS/"fig_dashboard.pdf",bbox_inches="tight")
fig.savefig(FIGS/"fig_dashboard.png",bbox_inches="tight"); plt.close(fig)

def plen(u): return len(getattr(uf.safe_parse(u),"path","") or "")
fig, axes = plt.subplots(2,3,figsize=(5.2,3.1),sharey=True)
bins=np.arange(0,81,4)
for ax,key in zip(axes.flat,ORD):
    df=CORPORA[key]; pl=df.url.map(plen); yy=df.label
    ax.hist(np.clip(pl[yy==0],0,80),bins=bins,alpha=.8,density=True,color=LEG)
    ax.hist(np.clip(pl[yy==1],0,80),bins=bins,alpha=.6,density=True,color=PHI)
    ax.set_title(NAMES[key],fontsize=7.5); ax.tick_params(labelsize=6)
axes.flat[5].axis("off")
axes.flat[5].legend(handles=[plt.Rectangle((0,0),1,1,color=LEG,alpha=.8),
                             plt.Rectangle((0,0),1,1,color=PHI,alpha=.6)],
                    labels=["Legitimate","Phishing"],loc="center",fontsize=8)
for ax in list(axes[1,:2])+[axes[0,2]]: ax.set_xlabel("URL path length",fontsize=7)
for ax in axes[:,0]: ax.set_ylabel("density",fontsize=7)
fig.tight_layout(pad=0.4)
fig.savefig(FIGS/"fig_artefact.pdf",bbox_inches="tight")
fig.savefig(FIGS/"fig_artefact.png",bbox_inches="tight"); plt.close(fig)

piv2 = merged.pivot_table(index="train",columns="test",values="roc_auc").reindex(index=ORD,columns=ORD)
L=[NAMES[k] for k in ORD]
fig,ax=plt.subplots(figsize=(3.7,3.2))
im=ax.imshow(piv2.values,cmap="RdYlGn",vmin=0,vmax=1,aspect="equal")
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(L,rotation=35,ha="right",fontsize=6); ax.set_yticklabels(L,fontsize=6); ax.grid(False)
for i in range(5):
    for j in range(5):
        v=piv2.values[i,j]
        ax.text(j,i,f"{v:.3f}",ha="center",va="center",fontsize=6,
                fontweight="bold" if v<0.5 else "normal")
        if v<0.5: ax.add_patch(plt.Rectangle((j-.5,i-.5),1,1,fill=False,edgecolor="blue",lw=1.6))
ax.set_xlabel("Evaluation dataset",fontsize=7); ax.set_ylabel("Training dataset",fontsize=7)
cb=plt.colorbar(im,ax=ax,fraction=0.046,pad=0.04); cb.ax.tick_params(labelsize=6); cb.set_label("ROC-AUC",fontsize=6.5)
fig.tight_layout(pad=0.3)
fig.savefig(FIGS/"fig_heatmap.pdf",bbox_inches="tight")
fig.savefig(FIGS/"fig_heatmap.png",bbox_inches="tight"); plt.close(fig)
print("three figures written")

three figures written


## Cell 9 — Compare against the provisional draft, then bundle

The draft currently contains provisional values from a feasibility run. Your
values below replace them wherever they differ; counts and cell identities are
expected to match exactly.

In [9]:
PROVISIONAL = {
    "within_mean": 0.9720, "cross_mean": 0.6678, "gap": 0.3042,
    "n_inverted": 6, "min_transfer": 0.2020, "probe_phishstorm": 0.7105,
    "ebbu_within": 0.9796, "kaggle_to_ebbu": 0.2946, "ebbu_to_kaggle": 0.6653,
}
yours = {
    "within_mean": round(w.mean(),4), "cross_mean": round(cr.mean(),4),
    "gap": round(gap,4), "n_inverted": int((cm<0.5).sum()),
    "min_transfer": round(cm.min(),4),
    "probe_phishstorm": round(probe.roc_auc.mean(),4),
    "ebbu_within": round(piv.loc["ebbu2017","ebbu2017"],4),
    "kaggle_to_ebbu": round(piv.loc["kaggle_malicious","ebbu2017"],4),
    "ebbu_to_kaggle": round(piv.loc["ebbu2017","kaggle_malicious"],4),
}
print(f"{'quantity':20s} {'yours':>10s} {'draft':>10s}")
flags = 0
for k, pv in PROVISIONAL.items():
    yv = yours[k]
    if k == "n_inverted":
        ok = yv == pv
    else:
        ok = abs(yv - pv) <= 0.03
    flags += (not ok)
    print(f"{k:20s} {yv:>10} {pv:>10} {'' if ok else ' *DIFFERS*'}")
print()
print("All within tolerance — the draft's provisional values match your run."
      if flags == 0 else
      f"{flags} value(s) differ — the paper will be written with YOUR numbers.")

import shutil, json as js, platform
meta = {"xgboost": xgboost.__version__, "sklearn": sklearn.__version__,
        "python": platform.python_version(), "seeds": list(SEEDS), "cap": CAP}
(TABLES / "ext5_run_metadata.json").write_text(js.dumps(meta, indent=2))
bundle = REPO_DIR / "reports" / "ext5_bundle"
if bundle.exists(): shutil.rmtree(bundle)
(bundle / "tables").mkdir(parents=True); (bundle / "figures").mkdir()
for f in ["transfer_matrix_5way.csv","asymmetry_5way.csv","ext5_results.csv",
          "phishstorm_probe.csv","structural_audit_new.csv",
          "new_dataset_hashes.json","ext5_run_metadata.json"]:
    if (TABLES/f).exists(): shutil.copy2(TABLES/f, bundle/"tables"/f)
for f in ["fig_dashboard.pdf","fig_dashboard.png","fig_artefact.pdf",
          "fig_artefact.png","fig_heatmap.pdf","fig_heatmap.png"]:
    if (FIGS/f).exists(): shutil.copy2(FIGS/f, bundle/"figures"/f)
archive = shutil.make_archive(str(DRIVE_DIR / "ext5_bundle"), "zip", bundle)
print("\nbundle written to Drive:", archive)
print("Download ext5_bundle.zip and send it — then we write the paper from your numbers.")

quantity                  yours      draft
within_mean              0.9715      0.972 
cross_mean               0.6694     0.6678 
gap                      0.3021     0.3042 
n_inverted                    6          6 
min_transfer              0.202      0.202 
probe_phishstorm         0.7156     0.7105 
ebbu_within              0.9774     0.9796 
kaggle_to_ebbu           0.3001     0.2946 
ebbu_to_kaggle           0.6742     0.6653 

All within tolerance — the draft's provisional values match your run.

bundle written to Drive: /content/drive/MyDrive/research/phishing-detection/ext5_bundle.zip
Download ext5_bundle.zip and send it — then we write the paper from your numbers.


---
### Then we write

Send `ext5_bundle.zip`. The paper is finalized from what it contains: every
value, table, and figure comes from this run of yours; anything provisional in
the current draft that differs gets overwritten. The Word version and the
GitHub repository README are synced in the same pass.